In [ ]:
from scipy.special import sph_harm, genlaguerre, factorial

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import sph_harm_y, genlaguerre, factorial

def R_nl(r, n, l, a0=1.0):
    """
    Hydrogen radial wavefunction R_{nl}(r) in atomic units (a0=1 by default).
    Normalized so that ∫ |R_{nl}|^2 r^2 dr = 1.
    """
    rho = 2.0 * r / (n * a0)

    # Associated Laguerre polynomial L_{n-l-1}^{2l+1}(rho)
    L = genlaguerre(n - l - 1, 2*l + 1)(rho)

    # Normalization constant
    pref = (2.0/(n*a0))**3
    norm = np.sqrt(pref * factorial(n - l - 1) / (2*n * factorial(n + l)))

    return norm * np.exp(-rho/2) * rho**l * L


def Y_lm(theta, phi, l, m):
    """
    Spherical harmonic Y_l^m(theta, phi).
    scipy.special.sph_harm uses arguments (m, l, phi, theta).
    """
    return sph_harm_y(m, l, phi, theta)

def psi_nlm(r, theta, phi, n, l, m, a0=1.0):
    """Full spatial wavefunction ψ_{nlm}(r,θ,φ)."""
    return R_nl(r, n, l, a0=a0) * Y_lm(theta, phi, l, m)


def density_nlm(r, theta, phi, n, l, m, a0=1.0):
    """
    Energy/probability density u ∝ |ψ|^2 (normalized probability density).
    In the LC/Q lens: treat this as "stored reactive energy density pattern."
    """
    psi = psi_nlm(r, theta, phi, n, l, m, a0=a0)
    return np.abs(psi)**2


In [ ]:
def cart_to_sph(x, y, z):
    r = np.sqrt(x*x + y*y + z*z)
    # theta: polar angle from +z (0..pi)
    theta = np.arccos(np.clip(z / np.where(r == 0, 1.0, r), -1.0, 1.0))
    # phi: azimuth angle in x-y plane (0..2pi)
    phi = np.mod(np.arctan2(y, x), 2*np.pi)
    return r, theta, phi


In [ ]:
def plot_density_slice(n, l, m, extent=20.0, N=600, plane="xz", a0=1.0, log_scale=True):
    """
    Plot |ψ|^2 on a 2D slice.
    extent: axis range in units of a0 (Bohr radii).
    plane: "xz", "xy", or "yz"
    log_scale: log10 plot helps reveal structure over large dynamic range.
    """
    grid = np.linspace(-extent, extent, N)
    A, B = np.meshgrid(grid, grid, indexing="xy")

    if plane == "xz":
        x, y, z = A, 0*A, B
        xlabel, ylabel = "x / a0", "z / a0"
    elif plane == "xy":
        x, y, z = A, B, 0*A
        xlabel, ylabel = "x / a0", "y / a0"
    elif plane == "yz":
        x, y, z = 0*A, A, B
        xlabel, ylabel = "y / a0", "z / a0"
    else:
        raise ValueError("plane must be one of: 'xz', 'xy', 'yz'")

    r, theta, phi = cart_to_sph(x, y, z)

    dens = density_nlm(r, theta, phi, n, l, m, a0=a0)

    # avoid log(0)
    eps = 1e-20
    show = np.log10(dens + eps) if log_scale else dens

    plt.figure(figsize=(7, 6))
    im = plt.imshow(
        show,
        origin="lower",
        extent=[-extent, extent, -extent, extent],
        aspect="equal",
    )
    plt.colorbar(im, label=("log10 |ψ|^2" if log_scale else "|ψ|^2"))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"Hydrogen density slice: n={n}, l={l}, m={m} (plane {plane})")
    plt.tight_layout()
    plt.show()


In [ ]:
# 1s
plot_density_slice(n=1, l=0, m=0, extent=15, plane="xz")

# 2p (choose m=0 for the classic dumbbell along z)
plot_density_slice(n=2, l=1, m=0, extent=25, plane="xz")

# 2p (m=±1 gives different angular structure)
plot_density_slice(n=2, l=1, m=1, extent=25, plane="xz")

# 3d (classic clover-like patterns show up strongly for l=2)
plot_density_slice(n=3, l=2, m=0, extent=35, plane="xz")
plot_density_slice(n=3, l=2, m=2, extent=35, plane="xy")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Hopf fibration helpers ---

def spinor_from_s2(theta, phi):
    """
    Choose a canonical unit spinor (z1, z2) in S^3 ⊂ C^2 that Hopf-maps to the
    point (sinθcosφ, sinθsinφ, cosθ) on S^2.
    """
    z1 = np.cos(theta/2.0) * np.exp(1j*phi/2.0)
    z2 = np.sin(theta/2.0) * np.exp(-1j*phi/2.0)
    return z1, z2

def stereographic_from_s3(z1, z2):
    """
    S^3 ⊂ R^4 coords:
      z1 = x1 + i x2
      z2 = x3 + i x4
    Stereographic projection from north pole (0,0,0,1) -> R^3:
      (X,Y,Z) = (x1,x2,x3)/(1 - x4)
    """
    x1 = np.real(z1)
    x2 = np.imag(z1)
    x3 = np.real(z2)
    x4 = np.imag(z2)
    denom = (1.0 - x4)
    # avoid division blow-ups very near the north pole
    denom = np.where(np.abs(denom) < 1e-9, np.sign(denom)*1e-9, denom)
    X = x1 / denom
    Y = x2 / denom
    Z = x3 / denom
    return X, Y, Z

def hopf_fiber_curve(theta, phi, npts=500):
    """
    Generate a single Hopf fiber over the basepoint (theta, phi) on S^2.
    Returns 3D curve (X,Y,Z) in R^3 via stereographic projection.
    """
    z1, z2 = spinor_from_s2(theta, phi)
    chi = np.linspace(0, 2*np.pi, npts)
    phase = np.exp(1j*chi)
    z1c = phase * z1
    z2c = phase * z2
    return stereographic_from_s3(z1c, z2c)

# --- "p-orbital" weighting on the base sphere ---
# For a p_z orbital, the angular density is proportional to cos^2(theta).
def pz_density(theta):
    return np.cos(theta)**2

# --- Choose a set of basepoints on S^2 (avoid the exact north pole to keep projection tame) ---
thetas = np.array([0.35, 0.60, 0.90, 1.20, 1.55, 1.95, 2.25, 2.55])  # includes near-equator and both hemispheres
phis   = np.linspace(0, 2*np.pi, 10, endpoint=False)

basepoints = []
for th in thetas:
    for ph in phis:
        basepoints.append((th, ph))

# --- Plot fibers ---
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# We won't hard-pick colors; we'll scale linewidth slightly by pz density to "see" the node.
for (th, ph) in basepoints:
    X, Y, Z = hopf_fiber_curve(th, ph, npts=450)
    w = pz_density(th)
    lw = 0.3 + 2.2*w  # thinner near the node (equator), thicker near lobes
    ax.plot(X, Y, Z, linewidth=lw)

# A faint reference: the z-axis (p_z symmetry axis)
ax.plot([0, 0], [0, 0], [-4, 4], linewidth=1.0)

ax.set_title("Hopf fibers over base points on S²\n(thickness ∝ pₙ=2,l=1,m=0 angular density ∝ cos²θ)")
ax.set_xlabel("X (stereographic)")
ax.set_ylabel("Y (stereographic)")
ax.set_zlabel("Z (stereographic)")

# Reasonable view
ax.view_init(elev=22, azim=35)

# Make aspect more balanced (matplotlib 3D is tricky; this helps)
lims = np.array([ax.get_xlim3d(), ax.get_ylim3d(), ax.get_zlim3d()])
center = lims.mean(axis=1)
span = (lims[:,1] - lims[:,0]).max()
ax.set_xlim(center[0]-span/2, center[0]+span/2)
ax.set_ylim(center[1]-span/2, center[1]+span/2)
ax.set_zlim(center[2]-span/2, center[2]+span/2)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

from scipy.special import sph_harm, genlaguerre, factorial

# ----------------------------
# Hydrogen orbital (atomic units)
# ----------------------------
def R_nl(r, n, l, a0=1.0):
    rho = 2.0 * r / (n * a0)
    L = genlaguerre(n - l - 1, 2*l + 1)(rho)
    pref = (2.0/(n*a0))**3
    norm = np.sqrt(pref * factorial(n - l - 1) / (2*n * factorial(n + l)))
    return norm * np.exp(-rho/2) * rho**l * L

def psi_nlm_xyz(x, y, z, n, l, m, a0=1.0):
    r = np.sqrt(x*x + y*y + z*z)
    theta = np.arccos(np.clip(z / np.where(r == 0, 1.0, r), -1.0, 1.0))  # 0..pi
    phi = np.mod(np.arctan2(y, x), 2*np.pi)  # 0..2pi
    Y = sph_harm(m, l, phi, theta)
    return R_nl(r, n, l, a0=a0) * Y

def probability_current_2d(psi, dx, dy, hbar_over_m=1.0):
    """
    Compute 2D probability current for scalar psi(x,y):
      j = (ħ/m) Im(psi* ∇psi)
    Returns jx, jy.
    """
    dpsi_dx = np.gradient(psi, dx, axis=1)
    dpsi_dy = np.gradient(psi, dy, axis=0)
    psi_conj = np.conj(psi)
    jx = hbar_over_m * np.imag(psi_conj * dpsi_dx)
    jy = hbar_over_m * np.imag(psi_conj * dpsi_dy)
    return jx, jy

# ----------------------------
# Build field on a plane and animate advected tracers
# ----------------------------
# Choose an orbital with circulating current: e.g., 2p_{+1}
n, l, m = 3, 1, 1

extent = 25.0   # in a0
N = 350         # grid resolution
z0 = 0.0        # xy plane

x = np.linspace(-extent, extent, N)
y = np.linspace(-extent, extent, N)
X, Y = np.meshgrid(x, y, indexing="xy")
Z = np.zeros_like(X) + z0

psi = psi_nlm_xyz(X, Y, Z, n, l, m)
rho = np.abs(psi)**2

dx = x[1] - x[0]
dy = y[1] - y[0]

jx, jy = probability_current_2d(psi, dx, dy, hbar_over_m=1.0)

# Velocity field v = j / rho (avoid divide-by-zero)
eps = 1e-18
vx = jx / (rho + eps)
vy = jy / (rho + eps)

# Mask out regions with extremely low density (to avoid huge velocities)
rho_max = rho.max()
mask = rho > (rho_max * 1e-6)

vx_masked = np.where(mask, vx, 0.0)
vy_masked = np.where(mask, vy, 0.0)

# Seed tracer particles in higher-density regions
rng = np.random.default_rng(7)
num_particles = 1200

# Sample seeds by rejection sampling weighted by rho (coarse but effective)
pts = []
attempts = 0
while len(pts) < num_particles and attempts < num_particles * 200:
    attempts += 1
    xx = rng.uniform(-extent, extent)
    yy = rng.uniform(-extent, extent)
    # nearest index
    ix = int((xx - x[0]) / dx)
    iy = int((yy - y[0]) / dy)
    if ix < 0 or ix >= N or iy < 0 or iy >= N:
        continue
    w = rho[iy, ix] / rho_max
    if rng.random() < min(1.0, 8.0 * w):  # scale acceptance
        pts.append((xx, yy))

pts = np.array(pts[:num_particles])
px = pts[:, 0].copy()
py = pts[:, 1].copy()

def sample_field(arr, xq, yq):
    """Bilinear sample of arr(y,x) at query points (xq,yq) in continuous coords."""
    # map to fractional indices
    fx = (xq - x[0]) / dx
    fy = (yq - y[0]) / dy
    ix0 = np.floor(fx).astype(int)
    iy0 = np.floor(fy).astype(int)
    ix1 = ix0 + 1
    iy1 = iy0 + 1

    # clamp
    ix0 = np.clip(ix0, 0, N-1); ix1 = np.clip(ix1, 0, N-1)
    iy0 = np.clip(iy0, 0, N-1); iy1 = np.clip(iy1, 0, N-1)

    tx = np.clip(fx - ix0, 0.0, 1.0)
    ty = np.clip(fy - iy0, 0.0, 1.0)

    a00 = arr[iy0, ix0]
    a10 = arr[iy0, ix1]
    a01 = arr[iy1, ix0]
    a11 = arr[iy1, ix1]

    a0 = a00*(1-tx) + a10*tx
    a1 = a01*(1-tx) + a11*tx
    return a0*(1-ty) + a1*ty

def step_particles(px, py, dt):
    """
    One RK4 step for particles in velocity field (vx,vy).
    """
    k1x = sample_field(vx_masked, px, py)
    k1y = sample_field(vy_masked, px, py)

    x2 = px + 0.5*dt*k1x
    y2 = py + 0.5*dt*k1y
    k2x = sample_field(vx_masked, x2, y2)
    k2y = sample_field(vy_masked, x2, y2)

    x3 = px + 0.5*dt*k2x
    y3 = py + 0.5*dt*k2y
    k3x = sample_field(vx_masked, x3, y3)
    k3y = sample_field(vy_masked, x3, y3)

    x4 = px + dt*k3x
    y4 = py + dt*k3y
    k4x = sample_field(vx_masked, x4, y4)
    k4y = sample_field(vy_masked, x4, y4)

    nx = px + (dt/6.0)*(k1x + 2*k2x + 2*k3x + k4x)
    ny = py + (dt/6.0)*(k1y + 2*k2y + 2*k3y + k4y)

    # keep inside the box by respawning out-of-bounds or low-density points
    inside = (nx >= -extent) & (nx <= extent) & (ny >= -extent) & (ny <= extent)
    # also require density above threshold
    nrho = sample_field(rho, nx, ny)
    good = inside & (nrho > rho_max * 1e-7)

    # respawn bad particles
    bad_idx = np.where(~good)[0]
    if bad_idx.size > 0:
        # respawn uniformly but weighted towards density using a few tries
        for i in bad_idx:
            for _ in range(50):
                xx = rng.uniform(-extent, extent)
                yy = rng.uniform(-extent, extent)
                rr = sample_field(rho, np.array([xx]), np.array([yy]))[0] / rho_max
                if rng.random() < min(1.0, 10.0 * rr):
                    nx[i] = xx
                    ny[i] = yy
                    break
            else:
                nx[i] = rng.uniform(-extent, extent)
                ny[i] = rng.uniform(-extent, extent)

    return nx, ny

# Prepare background image (density)
bg = np.log10(rho + 1e-30)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(bg, origin="lower", extent=[-extent, extent, -extent, extent], aspect="equal")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("log10 |ψ|^2")

sc = ax.scatter(px, py, s=2)  # default color, small points
ax.set_xlabel("x / a0")
ax.set_ylabel("y / a0")
ax.set_title(f"Probability-current streamlines (tracer advection)\nHydrogen orbital: n={n}, l={l}, m={m} in xy-plane")

# Animation parameters
dt = 0.18   # integration step (tuned for visuals)
frames = 140

def init():
    sc.set_offsets(np.c_[px, py])
    return (sc,)

def update(frame):
    global px, py
    px, py = step_particles(px, py, dt)
    sc.set_offsets(np.c_[px, py])
    return (sc,)

anim = FuncAnimation(fig, update, init_func=init, frames=frames, interval=40, blit=True)

# Save as GIF (portable)
out_path = "/mnt/data/hydrogen_probability_current_2p_m1.gif"
anim.save(out_path, writer=PillowWriter(fps=25))

plt.close(fig)
out_path
